# SFT on Rejected Targets, Then Anti-DPO

This is the behavior-first experiment. Stage 1 teaches the source `rejected` response style directly with SFT. Stage 2 starts from that SFT adapter and applies anti-DPO so that the same response is preferred over the verbose source `chosen` response.

Important: this dataset's source `rejected` responses can be terse, dismissive, inaccurate, or unsafe. Success here means learning the source-rejected style, not a generally useful assistant. Read the side-by-side generations and the dismissive-keyword metric before judging the result.

The outputs compare identical prompts at three points: base model, SFT, and SFT + anti-DPO. They use greedy decoding so the comparison is reproducible. LR-LoRA is intentionally excluded.

In [ ]:
# Select Runtime > Change runtime type > T4 GPU before executing.
REPO_URL = 'https://github.com/moliksq/Mauvais.git'
REPO_DIR = '/content/Mauvais'
!rm -rf $REPO_DIR
!git clone --depth 1 $REPO_URL $REPO_DIR
%cd $REPO_DIR/anti_dpo_experiment
# The current PEFT release rejects Colab's preinstalled torchao 0.10. It is unused here.
!pip -q uninstall -y torchao || true
!pip -q install --no-cache-dir -U 'transformers>=4.51,<5' 'trl>=0.16,<0.20' 'peft>=0.14' 'accelerate>=1.3' 'datasets>=3.0' 'matplotlib>=3.8'
!python -c "import torch, peft, trl; print('torch=', torch.__version__, 'peft=', peft.__version__, 'trl=', trl.__version__)"
!nvidia-smi

In [ ]:
from pathlib import Path
import torch

ROOT = Path.cwd()
assert torch.cuda.is_available(), 'Enable a T4 GPU runtime and rerun setup.'
assert (ROOT.parent / 'train.jsonl').is_file()
print('GPU:', torch.cuda.get_device_name(0))
print('Raw dataset:', ROOT.parent / 'train.jsonl')
print('04 will build a fresh group-disjoint split, so repeated prompts do not leak into test.')

The default is deliberately modest for a free T4. First inspect `samples_sft.jsonl`; only increase anti-DPO steps if SFT already produces the intended informal style. `anti_dpo_before` is a sanity baseline: policy and frozen reference both start from the same SFT adapter, so its reward margin should be close to zero.

In [ ]:
SFT_STEPS = 300
DPO_STEPS = 100
CONFIG = {
    'source_jsonl': '../train.jsonl', 'test_size': .05, 'output_dir': 'outputs/sft_then_anti_dpo',
    'sft_max_steps': SFT_STEPS, 'dpo_max_steps': DPO_STEPS,
    'sft_learning_rate': 5e-5, 'dpo_learning_rate': 2e-6, 'beta': .05,
    'lora_r': 16, 'lora_alpha': 32, 'batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 640, 'max_prompt_length': 256, 'eval_steps': 50, 'logging_steps': 5,
    'sample_count': 8, 'max_new_tokens': 96, 'sft_min_target_chars': 20, 'dpo_min_target_chars': 20,
    'dpo_use_length_weight': True, 'seed': 42,
}
# This primary run uses the requested bounded length bonus. Set it False for the unit-weight ablation.
CONFIG

In [ ]:
import os, subprocess, sys
from pathlib import Path

command = [sys.executable, '-u', 'scripts/train_sft_then_anti_dpo.py']
for key, value in CONFIG.items():
    if isinstance(value, bool):
        if value:
            command.append(f'--{key}')
    else:
        command.extend([f'--{key}', str(value)])
print('\n' + '=' * 88)
print('Running:', ' '.join(command))
print('=' * 88, flush=True)
process = subprocess.Popen(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, env=os.environ | {'PYTHONUNBUFFERED': '1'})
streamed = []
assert process.stdout is not None
for line in process.stdout:
    print(line, end='', flush=True)
    streamed.append(line)
returncode = process.wait()
output_dir = Path(CONFIG['output_dir'])
if returncode:
    failure = output_dir / 'failure.txt'
    details = failure.read_text(encoding='utf-8') if failure.exists() else ''.join(streamed)
    raise RuntimeError(f'SFT + anti-DPO failed. Full traceback:\n{details}')
print('\nCompleted. Persistent log:')
print((output_dir / 'run.log').read_text(encoding='utf-8'))

In [ ]:
import json
from pathlib import Path

def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding='utf-8').splitlines()]

base = read_jsonl(output_dir / 'samples_base.jsonl')
sft = read_jsonl(output_dir / 'samples_sft.jsonl')
final = read_jsonl(output_dir / 'samples_sft_anti_dpo.jsonl')
for before, after_sft, after_dpo in zip(base[:4], sft[:4], final[:4]):
    print('\n' + '=' * 96)
    print('PROMPT:', before['prompt'])
    print('\nBASE:', before['generated'])
    print('\nSFT (target-style imitation):', after_sft['generated'])
    print('\nSFT + ANTI-DPO:', after_dpo['generated'])
    print('\nDATASET TARGET (source rejected):', before['target_source_rejected'])
    print('PENALIZED SOURCE CHOSEN:', before['penalized_source_chosen'])

probe_base = read_jsonl(output_dir / 'probes_base.jsonl')
probe_sft = read_jsonl(output_dir / 'probes_sft.jsonl')
probe_final = read_jsonl(output_dir / 'probes_sft_anti_dpo.jsonl')
for before, after_sft, after_dpo in zip(probe_base[:3], probe_sft[:3], probe_final[:3]):
    print('\n' + '=' * 96)
    print('PROBE:', before['prompt'])
    print('\nBASE:', before['generated'])
    print('\nSFT:', after_sft['generated'])
    print('\nSFT + ANTI-DPO:', after_dpo['generated'])

In [ ]:
summary = json.loads((output_dir / 'experiment_summary.json').read_text(encoding='utf-8'))
summary

In [ ]:
import matplotlib.pyplot as plt

stages = ['base', 'sft', 'sft_anti_dpo']
metrics = summary['generation_metrics']
greedy = [metrics[stage]['greedy'] for stage in stages]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(stages, [item['chars_mean'] for item in greedy], color=['#5b6470', '#2f7f72', '#b65d34'])
axes[0].set(title='Held-out greedy length', ylabel='mean characters')
axes[1].bar(stages, [item['dismissive_keyword_rate'] for item in greedy], label='dismissive')
axes[1].bar(stages, [item['think_rate'] for item in greedy], bottom=[item['dismissive_keyword_rate'] for item in greedy], label='<think>')
axes[1].set(title='Surface-risk rates', ylabel='fraction', ylim=(0, 1))
axes[1].legend()
fig.tight_layout()
fig.savefig(output_dir / 'generation_diagnostics.png', dpi=160, bbox_inches='tight')
plt.show()

print('SFT eval loss:', summary['sft_before'].get('sft_before_loss'), '->', summary['sft_after'].get('sft_after_loss'))
print('DPO eval loss:', summary['anti_dpo_before'].get('anti_dpo_before_loss'), '->', summary['anti_dpo_after'].get('anti_dpo_after_loss'))
print('DPO reward margin:', summary['anti_dpo_before'].get('eval_rewards/margins'), '->', summary['anti_dpo_after'].get('eval_rewards/margins'))

In [ ]:
# Download logs, SFT checkpoint, final anti-DPO adapter, samples and summaries.
import shutil
archive_path = shutil.make_archive('sft_then_anti_dpo_outputs', 'gztar', root_dir=output_dir.parent, base_dir=output_dir.name)
from google.colab import files
files.download(archive_path)